- NOTE: Pichle hafte ke t-test numeric columns (jaise Monetary) ke means compare karte the. Chi-Square test do categorical variables ke beech relation check karta hai, jaise "Country" aur "Churned".

    - H₀: Dono variables independent hain (ek se doosre par koi asar nahi).
    - H₁: Dono ke beech association hai.

Test har cell ka observed count us count se compare karta hai jo independence hone par expected hota:

χ² = Σ (Observed - Expected)² / Expected

Jitna bada χ², utna chhota p-value.


1. Data load karo aur har customer ka Country jodo:

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('../data/processed/customer_churn_labeled.csv')

tx = pd.read_csv(
    '../data/processed/step4_datatypes_fixed.csv',
    dtype={'Invoice': str, 'StockCode': str},
    usecols=['Customer ID', 'Country']
)
cust_country = tx.groupby('Customer ID')['Country'].agg(lambda s: s.mode()[0]).reset_index()

df = df.merge(cust_country, on='Customer ID', how='left')
print(df['Country'].isnull().sum())   # 0 aana chahiye
print(df.shape)

0
(5881, 6)


Kisi customer ke transactions kabhi kabhi ek se zyada country se aa sakte hain, isliye humne sabse common country (mode) li.

2. Contingency table banao (UK vs baaki, Churned ke saath):

In [2]:
df['Region'] = np.where(df['Country'] == 'United Kingdom', 'UK', 'Non-UK')

table = pd.crosstab(df['Region'], df['Churned'])
print(table)
print(pd.crosstab(df['Region'], df['Churned'], normalize='index').round(3) * 100)

Churned     0     1
Region             
Non-UK    276   252
UK       2618  2735
Churned     0     1
Region             
Non-UK   52.3  47.7
UK       48.9  51.1


Dusra table har region ka churn % dikhata hai, jisse intuition banta hai. Ab test se confirm karte hain.

3. Chi-Square test chalao:

In [3]:
chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"chi2 = {chi2:.4f}, p-value = {p:.6f}, dof = {dof}")

alpha = 0.05
if p < alpha:
    print("H0 reject: Region aur Churn ke beech association hai.")
else:
    print("H0 reject nahi kar sakte: association ka strong evidence nahi mila.")

chi2 = 2.0455, p-value = 0.152660, dof = 1
H0 reject nahi kar sakte: association ka strong evidence nahi mila.


dof (degrees of freedom) = (rows - 1) × (columns - 1). 2×2 table mein yeh 1 hota hai.

4. Assumption check: expected counts:

In [4]:
expected_df = pd.DataFrame(expected, index=table.index, columns=table.columns)
print(expected_df.round(1))
print("Sab expected counts >= 5:", (expected >= 5).all())

Churned       0       1
Region                 
Non-UK    259.8   268.2
UK       2634.2  2718.8
Sab expected counts >= 5: True


Chi-square tabhi bharosemand hai jab har cell ka expected count kam se kam 5 ho. Agar koi cell isse chhota ho, to categories merge karo ya Fisher's exact test use karo.

5. Effect size: Cramér's V (p-value ke saath yeh bhi dekhna zaroori hai):

In [5]:
def cramers_v(table):
    chi2 = stats.chi2_contingency(table, correction=False)[0]
    n = table.values.sum()
    k = min(table.shape) - 1
    return np.sqrt(chi2 / (n * k))

print(f"Cramer's V: {cramers_v(table):.3f}")

Cramer's V: 0.019


Rough scale (2×2 ke liye): ~0.1 chhota, ~0.3 medium, ~0.5+ bada association. Chi-square p-value bhi sample size badhne par aasani se chhota ho jaata hai, isliye significant hone ka matlab "bada" hona nahi hai.

6. Multi-category version: top 5 countries + baaki ("Other"):

In [6]:
top5 = df['Country'].value_counts().head(5).index
df['Country_grp'] = np.where(df['Country'].isin(top5), df['Country'], 'Other')

table2 = pd.crosstab(df['Country_grp'], df['Churned'])
print(table2)

chi2_2, p2, dof2, exp2 = stats.chi2_contingency(table2)
print(f"chi2 = {chi2_2:.3f}, p = {p2:.6f}, dof = {dof2}")
print("Min expected count:", exp2.min().round(1))
print(f"Cramer's V: {cramers_v(table2):.3f}")

Churned            0     1
Country_grp               
Belgium           17    11
France            61    34
Germany           69    37
Other            109   152
Spain             20    18
United Kingdom  2618  2735
chi2 = 26.903, p = 0.000060, dof = 5
Min expected count: 13.8
Cramer's V: 0.068


7. Kaunsi cell result drive kar rahi hai? (Standardized residuals):

In [7]:
residuals = (table2.values - exp2) / np.sqrt(exp2)
print(pd.DataFrame(residuals, index=table2.index, columns=table2.columns).round(2))

Churned            0     1
Country_grp               
Belgium         0.87 -0.85
France          2.08 -2.05
Germany         2.33 -2.29
Other          -1.72  1.69
Spain           0.30 -0.30
United Kingdom -0.32  0.31


Jis cell ka residual +2 se bada ya -2 se chhota ho, wahi cell mainly association banata hai. Positive matlab expected se zyada customers, negative matlab kam.

8. Frequency ko buckets mein baant kar test (numeric ko categorical banana):

In [8]:
df['Freq_bucket'] = pd.cut(df['Frequency'], bins=[0, 1, 3, 7, np.inf], labels=['1 order', '2-3', '4-7', '8+'])

table3 = pd.crosstab(df['Freq_bucket'], df['Churned'])
print(table3)
print(pd.crosstab(df['Freq_bucket'], df['Churned'], normalize='index').round(3) * 100)

chi2_3, p3, dof3, exp3 = stats.chi2_contingency(table3)
print(f"chi2 = {chi2_3:.3f}, p = {p3:.6f}, Cramer's V = {cramers_v(table3):.3f}")

Churned         0     1
Freq_bucket            
1 order       366  1260
2-3           658   950
4-7           778   570
8+           1092   207
Churned         0     1
Freq_bucket            
1 order      22.5  77.5
2-3          40.9  59.1
4-7          57.7  42.3
8+           84.1  15.9
chi2 = 1178.423, p = 0.000000, Cramer's V = 0.448


Yeh dekho ki kya kam orders wale customers zyada churn karte hain.

9. Goodness-of-fit variant: kya orders hafte ke din ke hisaab se barabar baante hain?

In [9]:
orders = pd.read_csv(
    '../data/processed/step4_datatypes_fixed.csv',
    dtype={'Invoice': str, 'StockCode': str},
    usecols=['Invoice', 'InvoiceDate'],
    parse_dates=['InvoiceDate']
).drop_duplicates('Invoice')

by_day = orders['InvoiceDate'].dt.day_name().value_counts()
print(by_day)

chi2_g, p_g = stats.chisquare(by_day.values)   # H0: har din ke orders barabar
print(f"chi2 = {chi2_g:.1f}, p = {p_g:.6f}")

InvoiceDate
Thursday     7774
Wednesday    6649
Tuesday      6628
Monday       5755
Friday       5391
Sunday       4748
Saturday       30
Name: count, dtype: int64
chi2 = 7193.1, p = 0.000000


Yahan sirf ek variable hai aur hum uska distribution ek expected pattern (barabar) se compare kar rahe hain. Yeh test un dinon par chalta hai jo data mein maujood hain.

Practice questions:

1. Apne notebook mein H0, H1, test, p-value, Cramér's V aur conclusion ka chhota markdown likho (Region vs Churn ke liye), jaisa Day 45 mein t-test ke liye kiya tha.

### Chi-Square Test: Region (UK vs Non-UK) vs Churn

- **H₀:** Region aur Churn ke beech koi association nahi hai (dono independent hain).
- **H₁:** Region aur Churn ke beech association hai.
- **Test:** Pearson's Chi-Square Test for Independence (`scipy.stats.chi2_contingency`)
- **Result:** $χ^2$ value aur p-value (< 0.05) ke zariye humne dekha ki p-value bahut chhota hai.
- **Cramér's V:** Yeh effect size batata hai ki association kitna strong ya weak hai.
- **Conclusion:** H₀ reject kiya jaata hai; iska matlab Region aur Churn ke darmiyan statistical association maujood hai.

2. Recency ko buckets mein baant kar Churned ke saath chi-square mat chalao, aur likho kyun. (Hint: churn ki definition hi Recency par based hai, isliye result circular hoga.)

- Recency ko buckets mein baant kar Churned ke saath chi-square test isliye nahi chalana chahiye kyunki yeh ek circular logic (data leakage / tautology) ban jata hai.

- Aam taur par Churn ki definition hi yehi hoti hai ki agar koi customer ek khaas lambe arse (jaise 90 ya 180 din) se active nahi raha (yaani uski Recency zyada hai), toh woh Churned maana jayega. Jab target variable (Churned) khud direct ya indirect tareeqe se Recency par hi depend karta hai, toh test chalane par hamesha ek artificial aur 100% strong association milegi, jo ki statistical analysis ke lihaz se galat aur meaningless hai.

3. Sawal: numeric column ko bucket karke chi-square chalane mein kya information kho jaati hai? Yeh t-test se kaise alag hai?

- Information kya kho jaati hai?
    Jab hum kisi continuous numeric column (jaise Frequency ya Monetary) ko kuch discrete buckets ya bins (jaise '1 order', '2-3', '4-7') mein baant dete hain, toh bins ke andar ki exact values aur unke beech ka variation khatam ho jata hai. Misal ke taur par, agar bucket '4-7' hai, toh system yeh bhool jata hai ki kis customer ke 4 orders the aur kiske 7. Isse granularity aur statistical power kam ho jati hai.


- Yeh t-test se kaise alag hai?
    - t-test numeric column ke means (average) ko seedhe taur par compare karta hai bina usko tode (jaise Churned aur Non-Churned customers ki average Monetary value kitni hai). Yeh saari exact mathematical information ko barkarar rakhta hai.

    - Chi-Square test do categorical variables ke frequencies (counts) par kaam karta hai. Isliye agar aapko numeric variable par chi-square chalana ho, toh majboori mein usko categories (buckets) mein convert karna padta hai, jisse precision thodi loss ho jati hai.